In [1]:
!python -m spacy download en_core_web_trf
!pip install spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 27.0 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 25.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [en-core-web-trf]
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from collections import defaultdict
import string
import spacy
import nltk
import re
import math
import numpy as np

from collections import defaultdict
from collections import Counter

from sklearn.cluster import MiniBatchKMeans, KMeans
from scipy.sparse import lil_matrix, csr_matrix, coo_matrix
from scipy import spatial
from scipy.spatial.distance import euclidean
from sklearn.preprocessing import normalize

nltk.download('punkt_tab')
from nltk.corpus import stopwords
nltk.download('stopwords')



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
from google.colab import files
uploaded = files.upload()

Saving BUZZ1990.txt to BUZZ1990.txt
Saving BUZZ1980.txt to BUZZ1980.txt
Saving BUZZ2000.txt to BUZZ2000.txt
Saving BUZZ2010.txt to BUZZ2010.txt
Saving BUZZ2020.txt to BUZZ2020.txt
Saving CWAR2020.txt to CWAR2020.txt
Saving CWAR2010.txt to CWAR2010.txt
Saving CWAR2000.txt to CWAR2000.txt
Saving CWAR1990.txt to CWAR1990.txt
Saving CWAR1980.txt to CWAR1980.txt
Saving IDEOLOGY1980.txt to IDEOLOGY1980.txt
Saving IDEOLOGY1990.txt to IDEOLOGY1990.txt
Saving IDEOLOGY2000.txt to IDEOLOGY2000.txt
Saving IDEOLOGY2010.txt to IDEOLOGY2010.txt
Saving IDEOLOGY2020.txt to IDEOLOGY2020.txt
Saving POL2020.txt to POL2020.txt
Saving POL2000.txt to POL2000.txt
Saving POL2010.txt to POL2010.txt
Saving POL1990.txt to POL1990.txt
Saving POL1980.txt to POL1980.txt
Saving GEO2020.txt to GEO2020.txt
Saving GEO1990.txt to GEO1990.txt
Saving GEO2000.txt to GEO2000.txt
Saving GEO2010.txt to GEO2010.txt
Saving GEO1980.txt to GEO1980.txt


In [10]:

def parse_corpora(name, filter_contentwords=True):
    word_corpus = []
    filter_set = set(stopwords.words('english'))
    content_words = []

    with open(f"{name}", 'r', encoding='utf-8') as file:
        raw = file.read()

    # Split by the document delimiter
    articles = raw.split('____________________________________________________________')

    for article in articles:
        if not article.strip():
            continue

        # Extract the main body of text, stopping at any of the metadata headers
        body_match = re.search(r'(?:Full text:|Abstract:)(.*?)(?=Subject:|Location:|People:|_{10}|$)', article, re.DOTALL | re.IGNORECASE)
        body = body_match.group(1).strip() if body_match else ""

        if not body:
            continue

        # Pre-process text to replace hyphens and weird punctuation with spaces so words don't merge
        cleaned_body = re.sub(r'[-–—/]', ' ', body.lower())
        # Keep only alphanumeric characters and spaces
        cleaned_body = re.sub(r'[^a-z0-9\s]', '', cleaned_body)

        # Now split on whitespace safely
        normalized_words_in_body = cleaned_body.split()

        for word in normalized_words_in_body:
            # Filter out purely numerical digits or short noise
            if word.isdigit() or len(word) < 2:
                continue
            if word not in filter_set and filter_contentwords:
                content_words.append(word)
            word_corpus.append(word)

    return word_corpus, content_words

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
def compare_across_decades(decade_data, cat, intrigue_words, top_n=10):
    target_words = intrigue_words[cat]
    decades = sorted(decade_data.keys())

    print(f"{BOLD}Observed Cross-Decade Shift: {cat}{RESET}")

    for word in target_words:
        print(f"\n  {BOLD}'{word}'{RESET}")

        d1 = decades[0]
        for i in range(len(decades) - 1):
            d2 = decades[i+1]

            p1 = decade_data[d1]['profile']
            p2 = decade_data[d2]['profile']
            neighbors1 = set(w for w, _ in p1[word].most_common(top_n)) if word in p1 else set()
            neighbors2 = set(w for w, _ in p2[word].most_common(top_n)) if word in p2 else set()

            j = jaccard(neighbors1, neighbors2)
            if word in p1 and word in p2:
              cos_dist = cosine_distance_from_counters(p1[word], p2[word])
            else:
              cos_dist = None

            shared = neighbors1 | neighbors2

            v1 = np.array([p1[word].get(w, 0) if word in p1 else 0 for w in shared], dtype=np.float32)
            v2 = np.array([p2[word].get(w, 0) if word in p2 else 0 for w in shared], dtype=np.float32)
            dist = float(euclidean(v1, v2))

            print(f"    {d1}s to {d2}s:")
            print(f"      Cosine Distance : {cos_dist:.3f}" if cos_dist is not None else "      Cosine Distance : n/a")
            print(f"      Jaccard Score   : {j:.3f}")
            print(f"      Euclidean Dist  : {dist:.2f}")
            print(f"      Gained Neighbors: {', '.join(sorted(neighbors2 - neighbors1)) or 'none'}")
            print(f"      Lost Neighbors  : {', '.join(sorted(neighbors1 - neighbors2)) or 'none'}")


        for i in range(len(decades) - 1):
            d1, d2 = decades[i], decades[i+1]

            p1 = decade_data[d1]['profile']
            p2 = decade_data[d2]['profile']
            neighbors1 = set(w for w, _ in p1[word].most_common(top_n)) if word in p1 else set()
            neighbors2 = set(w for w, _ in p2[word].most_common(top_n)) if word in p2 else set()

            j = jaccard(neighbors1, neighbors2)
            if word in p1 and word in p2:
              cos_dist = cosine_distance_from_counters(p1[word], p2[word])
            else:
              cos_dist = None

            shared = neighbors1 | neighbors2

            v1 = np.array([p1[word].get(w, 0) if word in p1 else 0 for w in shared], dtype=np.float32)
            v2 = np.array([p2[word].get(w, 0) if word in p2 else 0 for w in shared], dtype=np.float32)
            dist = float(euclidean(v1, v2))

            print(f"    {d1}s to {d2}s:")
            print(f"      Cosine Distance : {cos_dist:.3f}" if cos_dist is not None else "      Cosine Distance : n/a")
            print(f"      Jaccard Score   : {j:.3f}")
            print(f"      Euclidean Dist  : {dist:.2f}")
            print(f"      Gained Neighbors: {', '.join(sorted(neighbors2 - neighbors1)) or 'none'}")
            print(f"      Lost Neighbors  : {', '.join(sorted(neighbors1 - neighbors2)) or 'none'}")


def jaccard(set1, set2):
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)

def cosine_distance_from_counters(c1, c2):
    if not c1 or not c2:
        return None
    dot = sum(v * c2.get(k, 0) for k, v in c1.items())
    n1 = math.sqrt(sum(v * v for v in c1.values()))
    n2 = math.sqrt(sum(v * v for v in c2.values()))
    if n1 == 0 or n2 == 0:
        return None
    return 1.0 - dot / (n1 * n2)

In [5]:
def print_relevant_clusters(X, labels, centroids, vocab, cat_words, n_clusters=25):
     cat_set = set(cat_words)
     relevant_found = 0

     for cluster_id in range(n_clusters):
         cluster_indices = np.where(labels == cluster_id)[0]
         cluster_vocab   = [vocab[i] for i in cluster_indices]
         target_indices = [i for i in cluster_indices if vocab[i] in cat_set]

         if not any(w in cat_set for w in cluster_vocab): # Skip clusters with no target words
             continue

         relevant_found += 1
         print(f"\n  Cluster {cluster_id}  ({len(cluster_vocab)} words):")


         for t_idx in target_indices:
            target_word = vocab[t_idx]
            print(f"  vs target '{target_word}':")

            sims = (X_norm[cluster_indices] @ X_norm[t_idx].T).toarray().flatten()
            order = np.argsort(-sims)  # descending: most similar first

            topn = int(10000 / n_clusters)

            for rank in order[:topn]:
                idx = cluster_indices[rank]
                word, sim = vocab[idx], sims[rank]
                if word in cat_set:
                    print(f"{RED}{BOLD}{word:15} {sim:.3f}{RESET}")
                else:
                    print("{:15} {:.3f}".format(word, sim))


     print(f"\n {relevant_found} / {n_clusters} clusters contained target words.")

In [6]:

def get_freq(content_words):
  freq = Counter(content_words)
  is_content_word = set(content_words)
  topw = {word: n for word, n in freq.items()}

  return is_content_word, topw


In [7]:
def contextualize(is_content_word, top10k, word_corpus, window=5):

  context_window = window
  profile = defaultdict(Counter)

  for i, token in enumerate(word_corpus):
      if token not in top10k:
          continue

      if i <= context_window:
          left_window  = word_corpus[0 : i]
      else:
          left_window  = word_corpus[i - context_window : i]

      right_window = word_corpus[i + 1 : i + 1 + context_window]
      for ctx_word in left_window + right_window:
          if ctx_word in is_content_word:
              profile[token][ctx_word] += 1

  return profile

In [8]:
from collections import Counter

def co_occurrence_profiles(initial_profile, intrigue_words):
  profile = initial_profile.copy()

  for word_phrase in intrigue_words:
      if ' ' in word_phrase:
          individual_words = word_phrase.split()

          all_words_in_profile = True
          for indiv_word in individual_words:
              if indiv_word not in profile:
                  all_words_in_profile = False
                  break

          if all_words_in_profile:
              combined_counter = Counter()
              for indiv_word in individual_words:
                  combined_counter += profile[indiv_word]
              profile[word_phrase] = combined_counter

  result_profile = {}
  for word in intrigue_words:
      normalized_word = word.lower()
      if normalized_word in profile:
          top_co_occurring = profile[normalized_word].most_common(20)
          result_profile[word] = Counter(dict(top_co_occurring))
      else:
          result_profile[word] = Counter()

  return result_profile

In [13]:


def counts_to_ppmi(X_counts):
    # Convert parse co-occurrence counts to PPMI with the same shape.
    X = X_counts.tocoo().astype(np.float64)

    total = float(X.data.sum())
    row_sum = np.asarray(X.sum(axis=1)).flatten()
    col_sum = np.asarray(X.sum(axis=0)).flatten()

    denominator = row_sum[X.row] * col_sum[X.col]  # > 0 for every nonzero entry
    pmi = np.log(X.data * total / denominator)
    pmi = np.maximum(pmi, 0.0)

    keep = pmi > 0
    ppmi = coo_matrix((pmi[keep], (X.row[keep], X.col[keep])), shape=X.shape)
    return ppmi.tocsr().astype(np.float32)


pref = [
    "GEO",
    "POL",
    "IDEOLOGY",
    "CWAR",
    "BUZZ"
]
dec = [
    "1980",
    "1990",
    "2000",
    "2010",
    "2020"
]
intrigue_words = {
    "GEO": [
        "china",
        "russia",
        "germany",
        "united kingdom",
        "european union",
        "india",
        "turkey",
        "saudi arabia"
    ],
    "POL": [
        "terrorism",
        "refugee",
        "hegemony",
        "collateral",
        "decoupling",
        "autocracy",
        "asymmetric",
        "deterrence",
        "proxy"
    ],
    "IDEOLOGY": [
        "populism",
        "environmentalism",
        "progressive",
        "traditional",
        "heritage",
        "tribalism",
        "nativist",
        "denier"
    ],
    "CWAR": [
        "woke",
        "diversity",
        "pronouns",
        "gender neutral",
        "racism",
        "merit",
        "equity"
    ],
    "BUZZ": [
        "mainstream",
        "pivot",
        "toxic",
        "weaponize",
        "framing"
    ]
}

RED = "\033[31m"
BOLD = "\033[1m"
RESET = "\033[0m"


pooled = Counter()
corpora = {}
contentlist = {}
iscontent = {}
topw = {}
pooled_content = set()

# We extract corpora for each category across each decade, and use it to build a global pooled corpus
# thus eliminating the need to perform procrustes alignment on the per-category-decade latent spaces in order to compare them.

for cat in pref:
    for decade in dec:
        id = f"{cat}{decade}"
        corpora[f"corp_{id}"], contentlist[f"content_{id}"] = parse_corpora(f"{id}.txt")  #extract raw corpus and content word filtered corpus
        iscontent[f"content_{id}"], topw[f"content_{id}"] = get_freq(contentlist[f"content_{id}"])  #convert the corpus to a sparse counter
        pooled.update(topw[f"content_{id}"])  # pool sparse counter corpus
        pooled_content.update(iscontent[f"content_{id}"]) #pool content word corpus

target_counts = {}
for cat in pref:
    for w in intrigue_words[cat]:
        target_counts[w] = pooled.get(w, 0)

pooled_vocab = {w: count for w, count in pooled.most_common(10000)}  # reduce pooled corpus counter to top 10k words only

for w, count in target_counts.items(): # if any target words are not in the top 10k most frequent words pooled corpus counter, add them back in with the correct counts
    if w not in pooled_vocab:
        pooled_vocab[w] = count


for cat in pref:
    print(f"{BOLD}Category: {cat}{RESET}")
    decade_data = {}
    for decade in dec:
        id = f"{cat}{decade}"
        profile = contextualize(pooled_content, pooled_vocab, corpora[f"corp_{id}"]) # iterating over each category and decade, contextualize each category-decade corpus, constructing embeddings, filtering out all non-content words.

        top_co_ocurring = co_occurrence_profiles(profile, intrigue_words[cat]) # use the previously generated embeddings to generate co-occurence profile counters for each targwt word
        for word, co_occurrences in top_co_ocurring.items():  # print out top co-occuring word profiles for each target word.
            print(f"\n{word}:")
            for co_word, count in co_occurrences.most_common(10):
                print(f"- {co_word}: {count}")


        word_to_index = {word: i for i, word in enumerate(pooled_vocab)} #get index of every word in pooled corpus
        vocab = list(word_to_index.keys())
        V = len(vocab)  # get dim of pooled corpus for constructing our PPMI matrix

        rows, cols, data = [], [], []
        for word, counter in profile.items():    # construct barebones sparse word co-occurence matrix
            i = word_to_index.get(word)
            if i is None:
                continue
            for coword, count in counter.items():
                j = word_to_index.get(coword)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
                    data.append(count)
        sparse_X = coo_matrix((data, (rows, cols)), shape=(V, V), dtype=np.float32).tocsr()


        X_ppmi = counts_to_ppmi(sparse_X) # convert our word co-occurence matrix to PPMI

        profile_ppmi = {}
        indptr = X_ppmi.indptr
        indices = X_ppmi.indices
        data = X_ppmi.data

        for i, word in enumerate(vocab): # copy our PPMI matrix back into a dictionary for easy traversal later
            s, e = indptr[i], indptr[i + 1]
            if e > s:
                profile_ppmi[word] = Counter({vocab[j]: float(v) for j, v in zip(indices[s:e], data[s:e])})
        decade_data[decade] = {"profile": profile_ppmi}

        nnz = np.asarray(X_ppmi.getnnz(axis=1)).flatten()  # eliminate words where PPMI = 0
        keep = np.where(nnz > 0)[0]
        X_filt = X_ppmi[keep]
        vocab_filt = [vocab[i] for i in keep]

        X_norm = normalize(X_filt, norm="l2", axis=1).tocsr()    # perform k-means clustering on our filtered PPMI matrix, then print clusters that contain our target words
        kmeans = KMeans(n_clusters=25, random_state=42, n_init=10)
        kmeans.fit(X_norm)
        labels = kmeans.labels_
        centroids = kmeans.cluster_centers_

        print_relevant_clusters(X_norm, labels, centroids, vocab, intrigue_words[cat])

    compare_across_decades(decade_data, cat, intrigue_words)





Streaming output truncated to the last 5000 lines.
chorus          0.002
withdrawing     0.002
join            0.002
custom          0.002
hitler          0.002
jinping         0.002
lies            0.002
teaching        0.002
letters         0.001
universities    0.001
hoped           0.001
franco          0.001
conclusion      0.001
romantic        0.001
drawing         0.001
invited         0.001
savage          0.001
opportunity     0.000
protestant      0.000
delivery        0.000
casting         0.000
compensation    0.000
uncertain       0.000
incentives      0.000
outraged        0.000
lonely          0.000
greatly         0.000
arrivals        0.000
slowed          0.000
julian          0.000
paramount       0.000
carr            0.000
oral            0.000
conn            0.000
sufficient      0.000
pocket          0.000
chicken         0.000
houston         0.000
districts       0.000
slide           0.000
preferred       0.000
slave           0.000
favored         0.000
str